# 5. Evaluate the best saved model

This runs after training. It reads the checkpoint selected by validation loss, rather than using the last epoch’s weights.

The default evaluates every validation recording. Set `RUN_FINAL_TEST = True` in setup only after selecting settings to score every held-out test recording.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import gc
import pandas as pd
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())

if "validation_df" not in globals():
    run_notebook("01_data.ipynb")

## Choose recordings and load the best checkpoint

Check that the saved split still matches the current data. Free the training model before loading another copy.

In [ ]:
EVALUATION_SPLIT = "test" if RUN_FINAL_TEST else "validation"
evaluation_df = test_df if RUN_FINAL_TEST else validation_df
checkpoint = RUN_DIR / "best"

if checkpoint.is_dir():
    saved_split = pd.read_csv(RUN_DIR / f"{EVALUATION_SPLIT}_split.csv", keep_default_na=False,
                             dtype={"audio_path": str, "text": str})
    pd.testing.assert_frame_equal(saved_split, evaluation_df[["audio_path", "text", "duration"]])
    saved_settings = json.loads((RUN_DIR / "settings.json").read_text())
    evaluation_language = saved_settings["language"]
    if "model" in globals():
        del model
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    processor = WhisperProcessor.from_pretrained(checkpoint, local_files_only=True)
    model = WhisperForConditionalGeneration.from_pretrained(checkpoint, local_files_only=True).to(DEVICE)
    model.config.use_cache = True
    model.eval()
else:
    print("No saved checkpoint yet. Complete training to evaluate recognition.")

## Generate transcripts and measure errors

WER counts word mistakes; CER counts character mistakes. They are fractions, so multiply by 100 for a percentage. Case, punctuation, and letters are kept as written in the dataset.

In [ ]:
if checkpoint.is_dir():
    progress = {"status": "evaluation", "split": EVALUATION_SPLIT,
                "recording": 0, "recordings": len(evaluation_df)}
    (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
    references = []
    predictions = []
    paths = []
    with torch.no_grad():
        for number, row in enumerate(evaluation_df.to_dict("records"), start=1):
            audio = read_audio(DATA_DIR / row["audio_path"])
            inputs = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE,
                                                 return_tensors="pt", return_attention_mask=True)
            inputs = {name: tensor.to(DEVICE) for name, tensor in inputs.items()}
            predicted_ids = model.generate(**inputs, language=evaluation_language, task="transcribe",
                                            max_new_tokens=MAX_NEW_TOKENS, return_timestamps=False)
            prediction = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True,
                                                       clean_up_tokenization_spaces=False)
            references.append(row["text"].strip())
            predictions.append(prediction.strip())
            paths.append(row["audio_path"])
            if number % 25 == 0:
                progress["recording"] = number
                (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
                print(f"Evaluated {number}/{len(evaluation_df)}", flush=True)

    results = {"split": EVALUATION_SPLIT, "count": len(references),
               "wer": wer(references, predictions), "cer": cer(references, predictions),
               "references": references, "predictions": predictions, "audio_paths": paths}
    (RUN_DIR / f"{EVALUATION_SPLIT}_metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
    print("WER:", results["wer"] * 100, "%")
    print("CER:", results["cer"] * 100, "%")
    for reference, prediction in zip(references[:5], predictions[:5]):
        print("Reference:", reference)
        print("Prediction:", prediction, "\n")
    progress.update(status="completed", recording=len(evaluation_df))
    (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))